# TrafficDemandElite995
**Competition-Grade Traffic Demand Prediction — Target R² ≥ 0.995**

---
Pipeline: Config → Validation → Load → Schema → EDA → Leakage → Clean → Features → Aggregation → Encoding → CatBoost → LightGBM → Optuna → OOF → Ensemble → Final Training → Feature Importance → Submission

## 1. Project Configuration

In [ ]:
import os, sys, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Add project root to path
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils import Config, validate_datasets, load_data, detect_schema, run_eda, clean_data, Timer, reduce_mem_usage
from src.feature_engineering import engineer_features
from src.encoding import apply_encodings
from src.aggregation import build_aggregation_features, build_memorization_features
from src.leakage_detection import detect_leakage
from src.training import (
    get_feature_columns, train_catboost_cv, train_lgbm_cv,
    optuna_catboost, optuna_lgbm,
    build_best_catboost_params, build_best_lgbm_params,
    predict_catboost, predict_lgbm,
    save_models, get_feature_importance
)
from src.ensemble import optimize_ensemble_weights, blend_predictions, generate_submission

Config.ensure_dirs()
Config.display()
print(f'\nPython {sys.version}')
print(f'NumPy {np.__version__}')
print(f'Pandas {pd.__version__}')

## 2. Dataset Validation

In [ ]:
validate_datasets()

## 3. Data Loading

In [ ]:
train_raw, test_raw = load_data()
print(f'\nTrain columns: {list(train_raw.columns)}')
print(f'Test columns:  {list(test_raw.columns)}')
train_raw.head()

## 4. Automatic Schema Detection

In [ ]:
schema = detect_schema(train_raw, test_raw)
TARGET = schema['target']
ID_COL = schema['id_column']
print(f'\nTarget: {TARGET}')
print(f'ID Column: {ID_COL}')

## 5. Exploratory Data Analysis

In [ ]:
run_eda(train_raw, test_raw, schema)

## 6. Leakage Detection

In [ ]:
leakage_findings = detect_leakage(train_raw, test_raw, schema)

## 7. Data Cleaning

In [ ]:
train, test, schema = clean_data(train_raw.copy(), test_raw.copy(), schema)
print(f'\nPost-cleaning missing values (train): {train.isnull().sum().sum()}')
print(f'Post-cleaning missing values (test): {test.isnull().sum().sum()}')

## 8. Feature Engineering

In [ ]:
with Timer('Feature Engineering'):
    train, test = engineer_features(train, test, schema)
print(f'\nTrain shape: {train.shape}')
print(f'Test shape: {test.shape}')

## 9. Aggregation Features

In [ ]:
with Timer('Aggregation Features'):
    train, test = build_aggregation_features(train, test, schema)
    train, test = build_memorization_features(train, test, schema)
print(f'\nTrain shape: {train.shape}')
print(f'Test shape: {test.shape}')

## 10. Target Encoding & Categorical Encoding

In [ ]:
with Timer('Encoding'):
    train, test = apply_encodings(train, test, schema)
print(f'\nTrain shape: {train.shape}')
print(f'Test shape: {test.shape}')

# Reduce memory
train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

## 11. Prepare Features for Modeling

In [ ]:
feature_cols = get_feature_columns(train, schema)
print(f'Number of features: {len(feature_cols)}')
print(f'Features: {feature_cols[:20]}...' if len(feature_cols) > 20 else f'Features: {feature_cols}')

y = train[TARGET].copy()
print(f'\nTarget stats:')
print(y.describe())

## 12. CatBoost Training (Default Params)

In [ ]:
with Timer('CatBoost Default CV'):
    cb_oof_default, cb_models_default, cb_scores_default = train_catboost_cv(
        train, y, feature_cols, n_folds=Config.N_FOLDS, use_gpu=Config.USE_GPU
    )
print(f'\nCatBoost Default CV R2: {np.mean(cb_scores_default):.6f}')

## 13. LightGBM Training (Default Params)

In [ ]:
with Timer('LightGBM Default CV'):
    lgb_oof_default, lgb_models_default, lgb_scores_default = train_lgbm_cv(
        train, y, feature_cols, n_folds=Config.N_FOLDS, use_gpu=Config.USE_GPU
    )
print(f'\nLightGBM Default CV R2: {np.mean(lgb_scores_default):.6f}')

## 14. Optuna Hyperparameter Optimization

In [ ]:
# CatBoost Optuna
with Timer('CatBoost Optuna'):
    cb_best_params, cb_best_score = optuna_catboost(
        train, y, feature_cols,
        n_trials=Config.OPTUNA_TRIALS,
        n_folds=Config.N_FOLDS,
        use_gpu=Config.USE_GPU
    )
print(f'\nCatBoost Optuna Best R2: {cb_best_score:.6f}')

In [ ]:
# LightGBM Optuna
with Timer('LightGBM Optuna'):
    lgb_best_params, lgb_best_score = optuna_lgbm(
        train, y, feature_cols,
        n_trials=Config.OPTUNA_TRIALS,
        n_folds=Config.N_FOLDS,
        use_gpu=Config.USE_GPU
    )
print(f'\nLightGBM Optuna Best R2: {lgb_best_score:.6f}')

## 15. OOF Prediction Generation (Optimized Params)

In [ ]:
# Retrain with best params
cb_opt_params = build_best_catboost_params(cb_best_params, use_gpu=Config.USE_GPU)
lgb_opt_params = build_best_lgbm_params(lgb_best_params, use_gpu=Config.USE_GPU)

print('CatBoost optimized params:'); print(json.dumps({k: str(v) for k,v in cb_opt_params.items()}, indent=2))
print('\nLightGBM optimized params:'); print(json.dumps({k: str(v) for k,v in lgb_opt_params.items()}, indent=2))

In [ ]:
with Timer('CatBoost Optimized CV'):
    cb_oof, cb_models, cb_scores = train_catboost_cv(
        train, y, feature_cols, n_folds=Config.N_FOLDS,
        params=cb_opt_params, use_gpu=Config.USE_GPU
    )

with Timer('LightGBM Optimized CV'):
    lgb_oof, lgb_models, lgb_scores = train_lgbm_cv(
        train, y, feature_cols, n_folds=Config.N_FOLDS,
        params=lgb_opt_params, use_gpu=Config.USE_GPU
    )

print(f'\nCatBoost  Optimized CV R2: {np.mean(cb_scores):.6f} +/- {np.std(cb_scores):.6f}')
print(f'LightGBM  Optimized CV R2: {np.mean(lgb_scores):.6f} +/- {np.std(lgb_scores):.6f}')

# Save OOF predictions
np.save(str(Config.PRED_DIR / 'cb_oof.npy'), cb_oof)
np.save(str(Config.PRED_DIR / 'lgb_oof.npy'), lgb_oof)
print('\nOOF predictions saved.')

## 16. Ensemble Weight Optimization

In [ ]:
best_weights, ensemble_r2 = optimize_ensemble_weights(
    [cb_oof, lgb_oof], y,
    model_names=['CatBoost', 'LightGBM']
)

print(f'\nFinal Ensemble R2: {ensemble_r2:.6f}')
print(f'Weights: CatBoost={best_weights[0]:.4f}, LightGBM={best_weights[1]:.4f}')

# Check if target score is met
if ensemble_r2 >= Config.TARGET_SCORE:
    print(f'\n  TARGET R2 >= {Config.TARGET_SCORE} ACHIEVED!')
else:
    print(f'\n  Current R2: {ensemble_r2:.6f}, Target: {Config.TARGET_SCORE}')

## 17. Final Training & Test Predictions

In [ ]:
# Generate test predictions
print('\n--- Generating Test Predictions ---')
cb_test_preds = predict_catboost(cb_models, test, feature_cols)
lgb_test_preds = predict_lgbm(lgb_models, test, feature_cols)

# Blend
final_preds = blend_predictions([cb_test_preds, lgb_test_preds], best_weights)

print(f'\nCatBoost test preds: mean={cb_test_preds.mean():.4f}, std={cb_test_preds.std():.4f}')
print(f'LightGBM test preds: mean={lgb_test_preds.mean():.4f}, std={lgb_test_preds.std():.4f}')
print(f'Ensemble test preds: mean={final_preds.mean():.4f}, std={final_preds.std():.4f}')

# Save models
save_models(cb_models, lgb_models, Config.MODEL_DIR)

# Save CV scores
cv_results = {
    'catboost_cv_scores': cb_scores,
    'catboost_mean_r2': float(np.mean(cb_scores)),
    'lightgbm_cv_scores': lgb_scores,
    'lightgbm_mean_r2': float(np.mean(lgb_scores)),
    'ensemble_r2': float(ensemble_r2),
    'ensemble_weights': best_weights,
    'catboost_best_params': {k: str(v) for k, v in cb_opt_params.items()},
    'lightgbm_best_params': {k: str(v) for k, v in lgb_opt_params.items()}
}
with open(Config.OUTPUT_DIR / 'cv_results.json', 'w') as f:
    json.dump(cv_results, f, indent=2)
print('\nCV results saved.')

## 18. Feature Importance

In [ ]:
importance_df = get_feature_importance(cb_models, lgb_models, feature_cols)
importance_df.to_csv(Config.OUTPUT_DIR / 'feature_importance.csv', index=False)
print('\nTop 15 Features:')
print(importance_df[['feature', 'combined']].head(15).to_string(index=False))

## 19. Submission Generation

In [ ]:
submission_path = Config.SUB_DIR / Config.SUBMISSION_FILENAME
submission = generate_submission(
    final_preds, test, ID_COL, 'demand', submission_path
)

# Also save to root
submission.to_csv(Config.PROJECT_ROOT / 'submission.csv', index=False)
print(f'\nSubmission also saved to: {Config.PROJECT_ROOT / "submission.csv"}')

print('\n' + '=' * 70)
print('  TrafficDemandElite995 - COMPLETE')
print('=' * 70)
print(f'  CatBoost CV R2:  {np.mean(cb_scores):.6f}')
print(f'  LightGBM CV R2:  {np.mean(lgb_scores):.6f}')
print(f'  Ensemble R2:     {ensemble_r2:.6f}')
print(f'  Weights:         CB={best_weights[0]:.4f}, LGB={best_weights[1]:.4f}')
print(f'  Submission rows: {len(submission)}')
print('=' * 70)